# Silver Cleaning (B): istinad və xarici data

**Məsuliyyətli:** Aslan

| Bronze | Silver |
|---|---|
| `bronze.olist_products_dataset` | `silver.products` |
| `bronze.product_category_name_translation` | `silver.category_translation` |
| `bronze.olist_sellers_dataset` | `silver.sellers` |
| `bronze.olist_geolocation_dataset` | `silver.geolocation` |
| `bronze.olist_order_reviews_dataset` | `silver.reviews` |
| `bronze.api_exchange_rates` | `silver.exchange_rates` |
| `bronze.api_weather` | `silver.weather` |





> `silver.reviews` üçün `silver.orders` (A tərəfindən) hazır olmalıdır.

In [8]:
import pandas as pd
from sqlalchemy import text
import sys
from pathlib import Path

# notebook-un olduğu qovluqdan bir yuxarı qovluq (db.py orada)
sys.path.append(str(Path.cwd().parent))

from db import get_engine

engine=get_engine()

In [9]:
"""SELECT sorğusu: nəticəni DataFrame kimi qaytarır."""
def q(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def run(sql):
    """CREATE/DROP/INSERT: icra edir və commit edir."""
    with engine.begin() as conn:
        conn.execute(text(sql))

In [3]:
# Təmizləmədən əvvəl bronze datasını yoxlayırıq

q("""
SELECT COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'bronze'
  AND TABLE_NAME = 'olist_products_dataset'
ORDER BY ORDINAL_POSITION
""")



,COLUMN_NAME,DATA_TYPE
0,product_id,varchar
1,product_category_name,varchar
2,product_name_lenght,varchar
3,product_description_lenght,varchar
4,product_photos_qty,varchar
5,product_weight_g,varchar
6,product_length_cm,varchar
7,product_height_cm,varchar
8,product_width_cm,varchar


### Silver Products

In [10]:
# Təmizləmədən əvvəl bronze datasını yoxlayırıq

q("""
SELECT COUNT(*) AS setir,                                   -- ümumi sətir sayı
       COUNT(DISTINCT product_id) AS unikal_id,             -- unikal məhsul sayı (dublikat varmı?)
       SUM(CASE WHEN product_category_name = '' THEN 1 ELSE 0 END) AS bos_kateqoriya,  -- boş kateqoriyalar
       SUM(CASE WHEN product_weight_g = '' THEN 1 ELSE 0 END) AS bos_ceki              -- boş çəkilər
FROM bronze.olist_products_dataset
""")

,setir,unikal_id,bos_kateqoriya,bos_ceki
0,32951,32951,610,2


In [ ]:
# Köhnə cədvəl varsa silirik ki, kodu təkrar işlətmək təhlükəsiz olsun
run("DROP TABLE IF EXISTS silver.products")

run("""
-- 1-ci mərhələ (c): təmizləmə və tip çevirmələri
WITH c AS (
    SELECT
        NULLIF(TRIM(product_id), '')                                    AS product_id,
        LOWER(NULLIF(TRIM(product_category_name), ''))                  AS category_name,
        -- bronze-da "lenght" yazılış səhvi var, silver-də "length" edirik
        TRY_CAST(NULLIF(TRIM(product_name_lenght), '') AS INT)          AS name_length,
        TRY_CAST(NULLIF(TRIM(product_description_lenght), '') AS INT)   AS description_length,
        TRY_CAST(NULLIF(TRIM(product_photos_qty), '') AS INT)           AS photos_qty,
        TRY_CAST(NULLIF(TRIM(product_weight_g), '') AS DECIMAL(10,2))   AS weight_g,
        TRY_CAST(NULLIF(TRIM(product_length_cm), '') AS DECIMAL(10,2))  AS length_cm,
        TRY_CAST(NULLIF(TRIM(product_height_cm), '') AS DECIMAL(10,2))  AS height_cm,
        TRY_CAST(NULLIF(TRIM(product_width_cm), '') AS DECIMAL(10,2))   AS width_cm
    FROM bronze.olist_products_dataset
),

-- 2-ci mərhələ (d): dublikatları nömrələyirik
d AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY (SELECT NULL)) AS rn
    FROM c
    WHERE product_id IS NOT NULL
)

SELECT product_id, category_name, name_length, description_length, photos_qty,
       weight_g, length_cm, height_cm, width_cm
INTO silver.products
FROM d
WHERE rn = 1
""")

In [ ]:
# Təmizləmədən sonra silver.products datasını yoxlayırıq
q("""
SELECT TOP 5* FROM silver.products
""")

,product_id,category_name,name_length,description_length,photos_qty,weight_g,length_cm,height_cm,width_cm
0,00066f42aeeb9f3007548bb9d3f33c38,perfumaria,53,596,6,300.0,20.0,16.0,16.0
1,00088930e925c41fd95ebfe695fd2655,automotivo,56,752,4,1225.0,55.0,10.0,26.0
2,0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,50,266,2,300.0,45.0,15.0,35.0
3,001795ec6f1b187d37335e1c4704762e,consoles_games,53,274,1,600.0,30.0,20.0,20.0
4,001b237c0e9bb435f2e54071129237e9,cama_mesa_banho,42,253,1,6000.0,40.0,4.0,30.0


### Silver Category translation

In [11]:
# Köhnə cədvəl varsa silirik
run("DROP TABLE IF EXISTS silver.category_translation")

run("""
SELECT DISTINCT                                                      -- DISTINCT dəqiq dublikatları silir
    LOWER(NULLIF(TRIM(product_category_name), ''))         AS category_name,          -- portuqal adı
    LOWER(NULLIF(TRIM(product_category_name_english), '')) AS category_name_english   -- ingilis adı
INTO silver.category_translation
FROM bronze.product_category_name_translation
WHERE NULLIF(TRIM(product_category_name), '') IS NOT NULL            -- kateqoriya adı boş olanları atırıq
""")